In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🔬 Hybrid Search Benchmark Lab

## What We're Building

A **head-to-head comparison** of three retrieval strategies on the **same dataset**:

| Strategy | How it works |
|----------|-------------|
| **BM25 (keyword)** | Classic term-frequency search — good for exact matches |
| **Dense retrieval** | Embedding similarity — good for semantic meaning |
| **Hybrid** | Combines BM25 + dense with Reciprocal Rank Fusion (RRF) |

## Why Benchmark?

Every blog says "hybrid is best" but **by how much?** And **when does each
method win?** We'll answer both with a real eval.

## Key Concept: Reciprocal Rank Fusion (RRF)

RRF combines two ranked lists by scoring each document:
```
score(d) = Σ 1 / (k + rank_i(d))   for each retriever i
```
where `k` is a smoothing constant (typically 60). Documents ranked high by
**either** retriever get a high fused score.

## Stack
- **rank_bm25** — BM25 keyword search
- **ChromaDB** — dense retrieval with embeddings
- **Custom RRF** — fusion implementation
- **Ollama** — local embeddings

## Step 1 — Imports & Setup

In [ ]:
# !pip install rank-bm25 langchain langchain-ollama chromadb -q

import time
from collections import defaultdict
from rank_bm25 import BM25Okapi
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Benchmark Dataset

We create a **knowledge base** and a set of **queries with known relevant documents**.
This lets us measure precision (did we find the right docs?).

In [ ]:
# Knowledge base — each doc has an ID for evaluation
corpus = [
    {"id": "D1", "text": "Python is a high-level programming language known for its readability. It supports multiple paradigms including object-oriented, functional, and procedural programming. Created by Guido van Rossum in 1991."},
    {"id": "D2", "text": "JavaScript is a scripting language used for web development. It runs in browsers and on servers via Node.js. It supports event-driven, functional, and prototype-based programming."},
    {"id": "D3", "text": "Machine learning is a subset of artificial intelligence where models learn from data. Common algorithms include decision trees, support vector machines, and neural networks."},
    {"id": "D4", "text": "Deep learning uses multi-layer neural networks to learn complex patterns. Popular frameworks include TensorFlow, PyTorch, and JAX. It excels in image recognition, NLP, and speech processing."},
    {"id": "D5", "text": "PostgreSQL is an open-source relational database management system. It supports ACID transactions, JSON storage, full-text search, and advanced indexing including GiST and GIN indexes."},
    {"id": "D6", "text": "MongoDB is a NoSQL document database that stores data in BSON format. It offers horizontal scaling through sharding, replica sets for high availability, and a flexible schema design."},
    {"id": "D7", "text": "Docker containers package application code with dependencies into portable units. Containers share the host OS kernel, making them lighter than virtual machines. Docker Compose orchestrates multi-container apps."},
    {"id": "D8", "text": "Kubernetes (K8s) orchestrates containerized workloads. It handles auto-scaling, service discovery, load balancing, and rolling deployments. Managed K8s is available on AWS EKS, GCP GKE, and Azure AKS."},
    {"id": "D9", "text": "REST APIs use HTTP methods (GET, POST, PUT, DELETE) for CRUD operations. They follow stateless client-server architecture. Common serialization formats include JSON and XML."},
    {"id": "D10", "text": "GraphQL is a query language for APIs that lets clients request exactly the data they need. It uses a typed schema, resolvers, and a single endpoint instead of multiple REST endpoints."},
    {"id": "D11", "text": "Git is a distributed version control system for tracking code changes. It supports branching, merging, and rebasing. Remote hosting is available on GitHub, GitLab, and Bitbucket."},
    {"id": "D12", "text": "Transformers are neural network architectures using self-attention mechanisms. Introduced in 'Attention Is All You Need' (2017). GPT, BERT, and T5 are all transformer-based models."},
    {"id": "D13", "text": "Retrieval-Augmented Generation (RAG) combines a retriever and a generator. The retriever finds relevant documents, and the generator produces an answer using those documents as context."},
    {"id": "D14", "text": "Vector databases store high-dimensional embeddings for similarity search. Examples include Pinecone, Weaviate, Qdrant, Milvus, and ChromaDB. They use approximate nearest neighbor (ANN) algorithms."},
    {"id": "D15", "text": "BM25 (Best Matching 25) is a probabilistic ranking function for keyword search. It considers term frequency, inverse document frequency, and document length normalization."},
]

# Queries with ground-truth relevant document IDs
benchmark_queries = [
    {"query": "What programming language did Guido van Rossum create?", "relevant": ["D1"], "type": "exact_match"},
    {"query": "How do neural networks learn complex patterns?", "relevant": ["D4", "D12"], "type": "semantic"},
    {"query": "database that supports JSON storage", "relevant": ["D5", "D6"], "type": "keyword"},
    {"query": "container orchestration and auto-scaling", "relevant": ["D7", "D8"], "type": "semantic"},
    {"query": "CRUD operations over HTTP", "relevant": ["D9"], "type": "keyword"},
    {"query": "What is the attention mechanism in AI?", "relevant": ["D12"], "type": "semantic"},
    {"query": "how does RAG combine retrieval with generation?", "relevant": ["D13", "D14"], "type": "semantic"},
    {"query": "BM25 ranking function term frequency", "relevant": ["D15"], "type": "exact_match"},
    {"query": "flexible schema NoSQL document store", "relevant": ["D6"], "type": "keyword"},
    {"query": "tools for managing code versions and branches", "relevant": ["D11"], "type": "semantic"},
]

print(f"Corpus: {len(corpus)} documents")
print(f"Queries: {len(benchmark_queries)} test queries")
print(f"  - exact_match: {sum(1 for q in benchmark_queries if q['type']=='exact_match')}")
print(f"  - keyword:     {sum(1 for q in benchmark_queries if q['type']=='keyword')}")
print(f"  - semantic:    {sum(1 for q in benchmark_queries if q['type']=='semantic')}")

## Step 3 — Build All Three Retrievers

In [ ]:
# === BM25 Retriever ===
tokenized_corpus = [doc["text"].lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def bm25_search(query: str, k: int = 5) -> list[dict]:
    """BM25 keyword search — returns ranked docs with scores."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"id": corpus[i]["id"], "score": float(scores[i]), "rank": rank + 1}
            for rank, i in enumerate(top_indices) if scores[i] > 0]


print("✅ BM25 retriever built")


# === Dense Retriever (ChromaDB + Ollama embeddings) ===
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
docs = [Document(page_content=d["text"], metadata={"id": d["id"]}) for d in corpus]
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="benchmark")


def dense_search(query: str, k: int = 5) -> list[dict]:
    """Dense embedding search via ChromaDB."""
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [{"id": doc.metadata["id"], "score": float(score), "rank": rank + 1}
            for rank, (doc, score) in enumerate(results)]


print("✅ Dense retriever built")


# === Hybrid Retriever (RRF fusion) ===
def reciprocal_rank_fusion(
    bm25_results: list[dict],
    dense_results: list[dict],
    k: int = 60,
    top_n: int = 5,
) -> list[dict]:
    """Fuse two ranked lists using Reciprocal Rank Fusion."""
    fused_scores = defaultdict(float)

    for result in bm25_results:
        fused_scores[result["id"]] += 1.0 / (k + result["rank"])

    for result in dense_results:
        fused_scores[result["id"]] += 1.0 / (k + result["rank"])

    sorted_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)[:top_n]
    return [{"id": doc_id, "score": fused_scores[doc_id], "rank": rank + 1}
            for rank, doc_id in enumerate(sorted_ids)]


def hybrid_search(query: str, k: int = 5) -> list[dict]:
    """Hybrid search = BM25 + Dense + RRF."""
    bm25_results = bm25_search(query, k=k)
    dense_results = dense_search(query, k=k)
    return reciprocal_rank_fusion(bm25_results, dense_results, top_n=k)


print("✅ Hybrid (RRF) retriever built")

## Step 4 — Evaluation Metrics

We'll measure:
- **Precision@k**: fraction of top-k results that are relevant
- **Recall@k**: fraction of relevant docs found in top-k
- **MRR (Mean Reciprocal Rank)**: 1/rank of the first relevant result

In [ ]:
def precision_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int = 5) -> float:
    """Fraction of retrieved docs that are relevant."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k


def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int = 5) -> float:
    """Fraction of relevant docs that were retrieved."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in relevant_ids if doc_id in top_k)
    return hits / len(relevant_ids) if relevant_ids else 0


def mrr(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """Reciprocal rank of the first relevant result."""
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0


print("Evaluation metrics defined")

## Step 5 — Run the Benchmark!

In [ ]:
K = 3  # Evaluate at top-3

results = {"BM25": [], "Dense": [], "Hybrid": []}
latencies = {"BM25": [], "Dense": [], "Hybrid": []}

for q in benchmark_queries:
    query = q["query"]
    relevant = q["relevant"]

    for name, search_fn in [("BM25", bm25_search), ("Dense", dense_search), ("Hybrid", hybrid_search)]:
        start = time.perf_counter()
        hits = search_fn(query, k=K)
        elapsed = time.perf_counter() - start

        retrieved_ids = [h["id"] for h in hits]
        results[name].append({
            "query": query,
            "type": q["type"],
            "precision": precision_at_k(retrieved_ids, relevant, K),
            "recall": recall_at_k(retrieved_ids, relevant, K),
            "mrr": mrr(retrieved_ids, relevant),
            "retrieved": retrieved_ids,
            "relevant": relevant,
        })
        latencies[name].append(elapsed)

print("Benchmark complete!")

In [ ]:
# === Aggregate Results ===
print(f"{'Metric':<15} {'BM25':>8} {'Dense':>8} {'Hybrid':>8}")
print("-" * 42)

for metric in ["precision", "recall", "mrr"]:
    row = []
    for name in ["BM25", "Dense", "Hybrid"]:
        avg = sum(r[metric] for r in results[name]) / len(results[name])
        row.append(avg)
    best = max(row)
    vals = ""
    for v in row:
        marker = " ★" if v == best else ""
        vals += f"{v:>7.3f}{marker} "
    print(f"{metric + '@' + str(K):<15} {vals}")

# Latency
print("-" * 42)
lat = ""
for name in ["BM25", "Dense", "Hybrid"]:
    avg_ms = sum(latencies[name]) / len(latencies[name]) * 1000
    lat += f"{avg_ms:>7.1f}  "
print(f"{'latency (ms)':<15} {lat}")

## Step 6 — Per-Query-Type Breakdown

The most interesting insight: **which method wins for which query type?**

In [ ]:
for qtype in ["exact_match", "keyword", "semantic"]:
    print(f"\n📊 Query Type: {qtype}")
    print(f"{'Method':<10} {'Precision':>10} {'Recall':>10} {'MRR':>8}")
    print("-" * 40)

    for name in ["BM25", "Dense", "Hybrid"]:
        type_results = [r for r in results[name] if r["type"] == qtype]
        if not type_results:
            continue
        avg_p = sum(r["precision"] for r in type_results) / len(type_results)
        avg_r = sum(r["recall"] for r in type_results) / len(type_results)
        avg_m = sum(r["mrr"] for r in type_results) / len(type_results)
        print(f"{name:<10} {avg_p:>10.3f} {avg_r:>10.3f} {avg_m:>8.3f}")

In [ ]:
# Detailed per-query view
print("\n📋 Per-Query Detail (Recall@3):\n")
for i, q in enumerate(benchmark_queries):
    bm25_r = results["BM25"][i]["recall"]
    dense_r = results["Dense"][i]["recall"]
    hybrid_r = results["Hybrid"][i]["recall"]
    best = max(bm25_r, dense_r, hybrid_r)
    winner = [n for n, v in [("BM25", bm25_r), ("Dense", dense_r), ("Hybrid", hybrid_r)] if v == best]
    print(f"  Q{i+1} [{q['type'][:7]:>7s}] BM25={bm25_r:.2f}  Dense={dense_r:.2f}  Hybrid={hybrid_r:.2f}  → {', '.join(winner)}")

## 🧠 Key Takeaways

| Finding | Explanation |
|---------|-------------|
| **BM25 excels at exact/keyword** | When query terms appear literally in the document |
| **Dense excels at semantic** | Understands meaning even with different vocabulary |
| **Hybrid is most consistent** | Rarely worst, often best — combines strengths |
| **RRF is simple but effective** | No training needed — just combine ranked lists |
| **BM25 is fastest** | No embedding computation needed |

## 🔧 Customization Ideas

- **Vary k**: How does performance change at k=1, 5, 10?
- **Weighted fusion**: Try `alpha * BM25_score + (1-alpha) * dense_score`
- **Larger corpus**: Test on 1000+ documents for realistic behavior
- **Cross-encoder reranker**: Add a reranker on top of hybrid for max quality